# PSC Robustness Analysis

In [1]:
def default_params(): 
    return {
        'current_model': 'M1', 
        'quantization': 'none', #['none',"int4", "int8", "float32", "float16"]
        'dataset': {
            'path': '/workspaces/CodeSmells/semeru-datasets/code_smells/generation',
            'generation_list': ['curated','greedy_search', 'beam_search', 'sampling', 'contrastive_search', 'top_k_sampling', 'top_p_sampling'],
            'excluded_generations' : [],
            'content_column': 'code',
            'sampling_size': 500,
        },
        'alignments_path': '/workspaces/CodeSmells/data/extension/generation/alignments',
        'robustness_results_path' : '/workspaces/CodeSmells/notebooks/extension/robustness_generation/07_robustness_analysis_results',
        'cache_dir': '/workspaces/CodeSmells/datax/hugging_face_cache',
    }
params = default_params()


#### Imports

In [2]:
import pandas as pd
import os
import time
import numpy as np
from datasets import load_dataset 
from statistics import mean, median
import json
import torch
import gc
import math
import scipy.stats as stats



In [3]:
import seaborn as sns; sns.set_theme()
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.backends.backend_pdf import PdfPages

In [4]:
from sklearn.preprocessing import StandardScaler
from scipy.stats import boxcox

#### Boostrapping

In [5]:
#| export
def bootstrapping( np_data, np_func, size ):
    """Create a bootstrap sample given data and a function
    For instance, a bootstrap sample of means, or mediands. 
    The bootstrap replicates are a long as the original size
    we can choose any observation more than once (resampling with replacement:np.random.choice)
    """
    
    #Cleaning NaNs
    #np_data_clean = np_data[ np.logical_not( np.isnan(np_data) ) ] 
    
    #The size of the bootstrap replicate is as big as size
    #Creating the boostrap replicates as long as the orignal data size
    #This strategy might work as imputation 
    bootstrap_repl = [ np_func( np.random.choice( np_data, size=len(np_data) ) ) for i in range( size ) ]
    
    #logging.info("Covariate: " + cov) #Empirical Mean
    #logging.info("Empirical Mean: " + str(np.mean(np_data_clean))) #Empirical Mean
    #logging.info("Bootstrapped Mean: " + str( np.mean(bootstrap_repl) ) ) #Bootstrapped Mean
    
    return np.array( bootstrap_repl )

#### Load Aggregates

In [6]:
def get_all_aggregates():
    aggregations = {}
    try:
        for generation in params['dataset']['generation_list']:
                generation_df = pd.read_json(f"{params['alignments_path']}/{params['current_model']}_q_{params['quantization']}/{generation}/aligned_smells.json")
                generation_df['generation'] = generation
                aggregations[generation] = generation_df
    except Exception:
         None
    return aggregations


In [7]:
aggregations = get_all_aggregates()
aggregations['base'] = aggregations.pop('curated')
aggregations['base']['generation'] = 'base'
aggregations.keys()

dict_keys(['greedy_search', 'beam_search', 'sampling', 'contrastive_search', 'top_k_sampling', 'top_p_sampling', 'base'])

In [8]:
aggregations['base']['s_msg_id'].unique()

array(['W0311', 'C0301', 'C0303', 'C0415', 'W0212', 'W0612', 'C0114',
       'W0613', 'C0116', 'C0304', 'C0305', 'C0123', 'C0103', 'W0622',
       'C0209', 'W0511', 'C0321', 'C0200', 'R1705', 'R0914', 'R1735',
       'R0912', 'W1514', 'C2801', 'R1710', 'C0325', 'W0621', 'R1720',
       'R1732', 'W0611', 'W0719', 'W0104', 'W0707', 'W0718', 'R0917',
       'R0913', 'W0102', 'W1309', 'W1406', 'C3001', 'W0601'], dtype=object)

#### Distribution comparisson between transformations

In [9]:
# 3. Define the logit transform
def logit_transform(x, eps=1e-9):
    x_clipped = np.clip(x, eps, 1 - eps)
    return np.log(x_clipped / (1 - x_clipped))

In [10]:
def logit_transform_psc(smell_msg_id, prob_column, decimal_spaces, aggregations):
    filtered_aggregations = {}
    for aggregation_name, aggregation_df in aggregations.items():
        filtered_df = aggregation_df[aggregation_df['s_msg_id'] == smell_msg_id].copy()

        # Apply logit transformation
        filtered_df[prob_column+'_trans'] = logit_transform(filtered_df[prob_column])

        ## Outlier removal after transformation
        # Calculate Q1 (25th percentile) and Q3 (75th percentile)
        Q1 = filtered_df[prob_column+'_trans'].quantile(0.25)
        Q3 = filtered_df[prob_column+'_trans'].quantile(0.75)

        # Compute the Interquartile Range (IQR)
        IQR = Q3 - Q1

        # Define lower and upper bounds for outliers
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR

        # Filter out the outliers
        filtered_df = filtered_df[(filtered_df[prob_column+'_trans'] >= lower_bound) & 
                                  (filtered_df[prob_column+'_trans'] <= upper_bound)]

        # Round to specified decimal places
        filtered_df[prob_column+'_trans'] = filtered_df[prob_column+'_trans'].round(decimal_spaces)

        filtered_aggregations[aggregation_name] = filtered_df
    return filtered_aggregations

In [29]:
def plot_density_distributions(smell_msg_id, psc_column, filtered_aggregations, ax, plot_stat= 'density'):
    combined_aggregates = pd.concat(filtered_aggregations.values(), ignore_index=True)
    # Plot histogram of actual data
    plt.figure(figsize=(10, 6))
    # Plot without the extra label parameter
    try: 
        sns.histplot(x=psc_column, 
                  data=combined_aggregates, 
                  hue='generation', 
                  kde=True, 
                  bins=500, 
                  stat=plot_stat, alpha= 0.3, ax=ax)
    except Exception as e: 
        print("Error in plot")
        sns.histplot(x=psc_column, 
             data=combined_aggregates, 
             hue='generation', 
             kde=False,  # <-- turn off KDE
             bins=500, 
             stat=plot_stat, alpha=0.3, ax=ax)

In [12]:
def compute_cliff_delta(distribution_1, distribution_2):
    n1, n2 = len(distribution_1), len(distribution_2)
    greater = sum(x > y for x in distribution_1 for y in distribution_2)
    lesser = sum(x < y for x in distribution_1 for y in distribution_2)
    return (greater - lesser) / (n1 * n2)

In [13]:
def bootstrap_ks(distribution_1, distribution_2, n_boot=1000):
    n1, n2 = len(distribution_1), len(distribution_2)
    D_boot = [stats.ks_2samp(np.random.choice(distribution_1, size=n1, replace=True),
                        np.random.choice(distribution_2, size=n2, replace=True))[0] 
              for _ in range(n_boot)]
    return np.percentile(D_boot, [2.5, 97.5])

In [14]:
def compare_distributions(distribution_1, distribution_2):
    np.random.seed(0)

    # Make distributions the same size
    min_size = min(len(distribution_1), len(distribution_2))
    if min_size <= 0: return None
    distribution_1 = np.random.choice(distribution_1, size=min_size, replace=False)
    distribution_2 = np.random.choice(distribution_2, size=min_size, replace=False)

    # Determine n_boot dynamically
    n_boot = max(100, min(1000, min_size // 10))
    
    ks_statistic, p_value = stats.ks_2samp(distribution_1, distribution_2, alternative='two-sided')
    st_effect_size = ks_statistic * math.sqrt((min_size * min_size) / (min_size + min_size))
    cliff_delta = compute_cliff_delta(distribution_1, distribution_2)
    wasserstein_dist = stats.wasserstein_distance(distribution_1, distribution_2)
    ci_ks_lower, ci_ks_upper = bootstrap_ks(distribution_1, distribution_2, n_boot)

    # Final decision based on all computed values
    practically_equivalent = (ci_ks_upper < 0.21) and (abs(cliff_delta) < 0.147) and (p_value > 0.05) and (wasserstein_dist < 0.1)
    
    return {
        "ks_stat": ks_statistic,
        "ci_ks_stat": (ci_ks_lower, ci_ks_upper),
        "p_value": p_value,
        "st_effect_size": st_effect_size,
        "cliffs_delta": cliff_delta,
        "wasserstein_dist" : wasserstein_dist,
        "equivalence": practically_equivalent
    }

In [15]:
def get_unique_smell_ids(aggregations): 
    # Get all unique smell_msg_id values from your aggregations dictionary
    unique_smell_ids = set()
    for aggregation_df in aggregations.values():
        unique_smell_ids.update(aggregation_df['s_msg_id'].unique())
    unique_smell_ids = list(unique_smell_ids)
    return unique_smell_ids

In [ ]:
def plot_transformation_distributions(prob_column, aggregations):
    # Get all unique smell_msg_id values from your aggregations dictionary
    unique_smell_ids = get_unique_smell_ids(aggregations)

    # Determine grid dimensions (using a square-like layout)
    n_plots = len(unique_smell_ids)
    ncols = math.ceil(math.sqrt(n_plots))
    nrows = math.ceil(n_plots / ncols)

    # Create the figure and axes using constrained_layout to automatically adjust spacing
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 5, nrows * 4), constrained_layout=True)
    axes = axes.flatten()  # Flatten for easier iteration


    # Loop over each smell_msg_id and plot in its corresponding subplot
    for i, smell_msg_id in enumerate(unique_smell_ids):
        # Transform data for current smell_msg_id
        filtered_aggregations = logit_transform_psc(smell_msg_id, prob_column, 4, aggregations)
        plot_density_distributions(smell_msg_id, prob_column+'_trans', filtered_aggregations, axes[i], 'density')
        # Set titles and labels for this subplot
        axes[i].set_title(f"Distributions of {smell_msg_id} by Generation Type")
        axes[i].set_xlabel("PSC (Propensity Smelly Score)")

    # Remove any unused subplots if the grid is larger than needed
    for j in range(i + 1, len(axes)):
        fig.delaxes(axes[j])
    

    fig.savefig(f"{params['robustness_results_path']}/{prob_column}.pdf", format="pdf")
    fig.savefig(f"{params['robustness_results_path']}/{prob_column}.png", format="png", dpi=300)
    
    # Close the figure to avoid displaying it
    plt.close(fig)


In [17]:
def perform_base_stat_tests(psc_column, aggregations):
    stat_test_df = pd.DataFrame(columns=['s_msg_id', 'generation', 'ks_stat', 'ci_ks_stat', 'p_value', 'st_effect_size', 'cliffs_delta', 'wasserstein_dist', 'equivalence'])
    for code_smell_id in get_unique_smell_ids(aggregations):
        for generation in [agg for agg in list(aggregations.keys()) if agg != 'base']:
            filtered_aggregations = logit_transform_psc(code_smell_id, psc_column, 4, aggregations)
            stat_test_results = compare_distributions(filtered_aggregations['base'][psc_column+'_trans'], filtered_aggregations[generation][psc_column+'_trans'])
            if stat_test_results is None: continue
            stat_test_results['generation'] = generation
            stat_test_results['s_msg_id'] = code_smell_id
            stat_test_df.loc[len(stat_test_df)] = stat_test_results
    return stat_test_df

In [18]:
def perform_anova(distribution_list):
    # Ensure input is a list of at least two distributions
    if len(distribution_list) < 2:
        raise ValueError("At least two distributions are required for ANOVA.")
    
    # Ensure all distributions have the same size
    min_size = min(len(distribution) for distribution in distribution_list)
    if min_size <= 0: return None
    distribution_list = [distribution[:min_size] for distribution in distribution_list]
    
    # Perform ANOVA test
    f_statistic, p_value = stats.f_oneway(*distribution_list)
    
    # Compute effect size (eta squared)
    mean_overall = np.mean(np.concatenate(distribution_list))
    
    sum_squares_between = sum(len(distribution) * (np.mean(distribution) - mean_overall) ** 2 for distribution in distribution_list)
    sum_squares_total = sum((sample - mean_overall) ** 2 for distribution in distribution_list for sample in distribution)
    eta_squared = sum_squares_between / sum_squares_total if sum_squares_total != 0 else 0
    
    return {
        'f_statistic': f_statistic,
        'p_value': p_value,
        'eta_squared': eta_squared
    }

In [19]:
def perform_anova_tests(psc_column, aggregations):
    stat_test_df = pd.DataFrame(columns=['s_msg_id', 'f_statistic', 'p_value', 'eta_squared'])
    for code_smell_id in get_unique_smell_ids(aggregations):
        filtered_aggregations = logit_transform_psc(code_smell_id, psc_column, 4, aggregations)
        stat_test_results = perform_anova([aggregation_df[psc_column+'_trans'] for aggregation_df in filtered_aggregations.values()])
        if stat_test_results is None: continue
        stat_test_results['s_msg_id'] = code_smell_id
        stat_test_df.loc[len(stat_test_df)] = stat_test_results
    return stat_test_df
    

#### Comparisons against base

In [20]:
aggregations['base'].columns

Index(['id', 'commit_id', 'repo', 'path', 'file_name', 'fun_name',
       'commit_message', 'code', 'url', 'language', 'ast_errors',
       'n_ast_errors', 'ast_levels', 'n_whitespaces', 'n_words', 'vocab_size',
       'complexity', 'nloc', 'token_counts', 'n_ast_nodes', 'n_identifiers',
       's_msg_id', 's_line', 's_column', 's_end_line', 's_end_column',
       's_code', 'category', 'input_lenght', 'prompt', 'contrastive_search',
       'top_k_sampling', 'top_p_sampling', 'input_ids', 'max_prob', 'min_prob',
       'actual_prob', 'loss', 'code_smell_pos', 'code_smell_actual_logits',
       'code_smell_max_logits', 'code_smell_min_logits',
       'code_smell_actual_prob_median', 'code_smell_max_prob_median',
       'code_smell_min_prob_median', 'code_smell_actual_prob_mean',
       'code_smell_max_prob_mean', 'code_smell_min_prob_mean',
       'code_smell_psc_entropy', 'code_smell_psc_relative', 'generation'],
      dtype='object')

In [21]:
psc_column = 'code_smell_psc_relative'
stat_tests_df = perform_base_stat_tests(psc_column, aggregations)

In [22]:
len(stat_tests_df[stat_tests_df['equivalence']==True])

31

In [23]:
stat_tests_df[(stat_tests_df['equivalence']==True) & (stat_tests_df['generation']=='sampling')]

,s_msg_id,generation,ks_stat,ci_ks_stat,p_value,st_effect_size,cliffs_delta,wasserstein_dist,equivalence
8,C0114,sampling,0.052521,"(0.046218487394957986, 0.12106092436974782)",0.528012,0.810255,-0.006378,0.031823,True
38,C0200,sampling,0.065574,"(0.06557377049180328, 0.16939890710382513)",0.827718,0.627250,0.015975,0.088624,True
68,C0305,sampling,0.026667,"(0.013333333333333334, 0.06174999999999997)",0.999932,0.326599,-0.009889,0.002958,True
110,W0102,sampling,0.072727,"(0.05670454545454545, 0.1638636363636363)",0.606838,0.762770,0.003244,0.062240,True
116,R0914,sampling,0.091483,"(0.06624605678233439, 0.17231861198738158)",0.140901,1.151737,0.039696,0.068081,True


#### Comparisons all included

In [24]:
aggregations['base'].columns

Index(['id', 'commit_id', 'repo', 'path', 'file_name', 'fun_name',
       'commit_message', 'code', 'url', 'language', 'ast_errors',
       'n_ast_errors', 'ast_levels', 'n_whitespaces', 'n_words', 'vocab_size',
       'complexity', 'nloc', 'token_counts', 'n_ast_nodes', 'n_identifiers',
       's_msg_id', 's_line', 's_column', 's_end_line', 's_end_column',
       's_code', 'category', 'input_lenght', 'prompt', 'contrastive_search',
       'top_k_sampling', 'top_p_sampling', 'input_ids', 'max_prob', 'min_prob',
       'actual_prob', 'loss', 'code_smell_pos', 'code_smell_actual_logits',
       'code_smell_max_logits', 'code_smell_min_logits',
       'code_smell_actual_prob_median', 'code_smell_max_prob_median',
       'code_smell_min_prob_median', 'code_smell_actual_prob_mean',
       'code_smell_max_prob_mean', 'code_smell_min_prob_mean',
       'code_smell_psc_entropy', 'code_smell_psc_relative', 'generation'],
      dtype='object')

In [ ]:
psc_column = 'code_smell_psc_relative'
filtered_generations = {k: v for k, v in aggregations.items() if k not in params['dataset']['excluded_generations']}
anova_results = perform_anova_tests(psc_column, filtered_generations)
anova_results = anova_results.sort_values(by=['f_statistic'], ascending=True)
anova_results

,s_msg_id,f_statistic,p_value,eta_squared
22,R0912,0.616064,7.175799e-01,0.005150
18,W0102,1.770529,1.016357e-01,0.007683
17,R0913,2.605228,1.613763e-02,0.007245
25,R0917,2.612433,1.587710e-02,0.007433
1,C0114,3.505455,1.850553e-03,0.006286
19,R0914,3.575089,1.605474e-03,0.014658
24,W0613,3.821293,8.426989e-04,0.006534
9,C2801,4.570065,1.488790e-04,0.040839
28,R1710,6.444701,9.663834e-07,0.019146
34,W0601,7.420063,7.359764e-08,0.025004


### Vissual Comparison 

In [31]:
psc_column = 'code_smell_psc_relative'
filtered_generations = {k: v for k, v in aggregations.items() if k not in params['dataset']['excluded_generations']}
plot_transformation_distributions(psc_column, filtered_generations)

Error in plot
Error in plot
Error in plot


/tmp/ipykernel_1866699/1399314462.py:4: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  plt.figure(figsize=(10, 6))


Error in plot
Error in plot
Error in plot
Error in plot
Error in plot
Error in plot
Error in plot
Error in plot
Error in plot
Error in plot
Error in plot
Error in plot
Error in plot
Error in plot
Error in plot
Error in plot
Error in plot
Error in plot
Error in plot
Error in plot
Error in plot
Error in plot
Error in plot
Error in plot
Error in plot
Error in plot


<Figure size 1000x600 with 0 Axes>

<Figure size 1000x600 with 0 Axes>

...

<Figure size 1000x600 with 0 Axes>

<Figure size 1000x600 with 0 Axes>

<Figure size 1000x600 with 0 Axes>

<Figure size 1000x600 with 0 Axes>

<Figure size 1000x600 with 0 Axes>

### Variability

In [33]:
## variability (std)
#['curated','greedy_search', 'beam_search', 'sampling', 'contrastive_search', 'top_k_sampling', 'top_p_sampling']
alignments_df = aggregations['greedy_search']
alignments_df.groupby('s_msg_id')['code_smell_psc_relative'].describe().sort_values(by='std', ascending=True)

,count,mean,std,min,25%,50%,75%,max
s_msg_id,,,,,,,,
W1510,2.0,0.928707,0.000000e+00,0.928707,0.928707,0.928707,0.928707,0.928707
E1130,2.0,1.000000,0.000000e+00,1.000000,1.000000,1.000000,1.000000,1.000000
W0109,4.0,0.977191,0.000000e+00,0.977191,0.977191,0.977191,0.977191,0.977191
C0414,7.0,1.000000,4.879501e-11,1.000000,1.000000,1.000000,1.000000,1.000000
E2511,9.0,1.000000,7.264832e-11,1.000000,1.000000,1.000000,1.000000,1.000000
...,...,...,...,...,...,...,...,...
W0133,1.0,0.781424,NaN,0.781424,0.781424,0.781424,0.781424,0.781424
W1116,1.0,1.000000,NaN,1.000000,1.000000,1.000000,1.000000,1.000000
W1303,1.0,0.694862,NaN,0.694862,0.694862,0.694862,0.694862,0.694862
